In [16]:
import os
import pandas as pd

output_folder = r"C:\Users\joaqu\OneDrive\Desktop\MAGISTER"

dfCongreso   = pd.read_excel(os.path.join(output_folder, "dfCongreso.xlsx"))
dfSecuencia  = pd.read_excel(os.path.join(output_folder, "dfSecuencia.xlsx"))
df_1_to_2    = pd.read_excel(os.path.join(output_folder, "df_1_to_2.xlsx"))
df_2_to_3    = pd.read_excel(os.path.join(output_folder, "df_2_to_3.xlsx"))
dfNivelEjercicio = pd.read_excel(os.path.join(output_folder, "dfNivelEjercicio.xlsx"))

In [17]:
dfCongreso

,userid,grupo_intervencion,num_steps,steps_primera,pasos_reales_ejercicio,numero_sesiones_diferentes,Tasa_exito,autoeficacia_general,progreso_inicial,progreso_inicial_grupo,...,proporcion_casos_dw,proporcion_pasos_dw,proporcion_tiempo_dw,preferencia_casos_comparacion,preferencia_casos_uw,preferencia_steps_uw,preferencia_tiempo_uw,preferencia_pasos_comparacion,tupdivtdown,tupdivtdownpasos
0,303,OSLM,2,2,2,1,1.000000,NaN,0.250000,-1.000000,...,0.000000,0.000000,0.000000,-1.000000,NaN,NaN,NaN,-1.000000,1.000000,1.000000
1,452,OSLM+VP,301,206,210,5,0.684385,59.464286,0.464938,0.308025,...,0.361446,0.398671,0.356453,0.807229,0.200000,0.090909,0.218873,0.754153,1.560146,1.198347
2,522,OSLM+VP,2,1,1,1,0.500000,NaN,0.230000,-1.000000,...,0.000000,0.000000,0.000000,-1.000000,NaN,NaN,NaN,-1.000000,1.000000,1.000000
3,1101,OSLM+VP,65,49,49,1,0.753846,NaN,0.383333,0.395833,...,0.708333,0.784615,0.794071,1.000000,-0.416667,-0.569231,-0.588143,1.000000,0.259968,0.288462
4,1134,OSLM,44,18,18,1,0.409091,NaN,0.300000,0.370000,...,1.000000,1.000000,1.000000,1.000000,-1.000000,-1.000000,-1.000000,1.000000,0.000518,0.022222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,2407,OSLM+VP,2,0,0,1,0.000000,100.000000,0.250000,0.270000,...,1.000000,1.000000,1.000000,1.000000,-1.000000,-1.000000,-1.000000,1.000000,0.011763,0.333333
177,2419,OSLM+VP,20,0,0,1,0.000000,78.333333,0.315000,0.245000,...,0.166667,0.100000,0.055991,1.000000,0.666667,0.800000,0.888019,1.000000,15.855958,6.333333
178,2424,OSLM+VP,10,0,0,1,0.000000,30.000000,0.250000,0.250000,...,0.000000,0.000000,0.000000,-1.000000,NaN,NaN,NaN,-1.000000,1.000000,1.000000
179,2434,OSLM+VP,6,0,0,1,0.000000,100.000000,0.250000,0.180000,...,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,137.107000,7.000000


In [19]:
import pandas as pd

df = dfSecuencia.loc[dfSecuencia["pollCount"].isin([1, 2, 3])].copy()

grupos = (df[["userid", "grupo_intervencion"]]
          .dropna()
          .drop_duplicates("userid"))

# 2) SE wide (SEt_1, SEt_2, SEt_f)
se = (df.pivot_table(index="userid", columns="pollCount", values="pollValue", aggfunc="mean")
        .reindex(columns=[1, 2, 3])
        .rename(columns={1: "SEt_1", 2: "SEt_2", 3: "SEt_f"}))

# 3) Deltas
se = se.assign(
    DeltaSEt_1_2=lambda x: x["SEt_2"] - x["SEt_1"],
    DeltaSEt_2_f=lambda x: x["SEt_f"] - x["SEt_2"],
    DeltaSEt_1_f=lambda x: x["SEt_f"] - x["SEt_1"],
)

vars_obj = ["SEt_1","SEt_2","SEt_f","DeltaSEt_1_2","DeltaSEt_2_f","DeltaSEt_1_f"]

# 4) Unir grupo y resumir por grupo (Media (SD))
df_est = (grupos.merge(se.reset_index(), on="userid", how="left")
                .dropna(subset=["grupo_intervencion"]))

mean_sd = lambda s: f"{s.mean():.2f} ({s.std(ddof=1):.2f})" if s.notna().any() else "—"

tabla = (df_est.groupby("grupo_intervencion")[vars_obj]
              .agg(mean_sd)
              .T)

tabla.index = tabla.index.map({
    "SEt_1": "SEt₁", "SEt_2": "SEt₂", "SEt_f": "SEt_f",
    "DeltaSEt_1_2": "∆SEt₁,₂", "DeltaSEt_2_f": "∆SEt₂,f", "DeltaSEt_1_f": "∆SEt₁,f",
})

print("\nP2. Variables de Autoeficacia (f = 3)")
print(tabla)



P2. Variables de Autoeficacia (f = 3)
grupo_intervencion            OLM           OSLM        OSLM+VP
SEt₁                53.92 (30.20)  62.72 (30.05)  58.22 (31.89)
SEt₂                70.27 (27.22)  74.76 (24.99)  68.57 (28.14)
SEt_f               63.89 (31.12)  74.43 (29.40)  82.80 (19.79)
∆SEt₁,₂             13.51 (15.06)  12.05 (13.19)  13.91 (15.72)
∆SEt₂,f              3.22 (13.54)   4.47 (22.84)  12.50 (15.30)
∆SEt₁,f             14.74 (20.62)  20.35 (26.41)  27.95 (24.50)


In [20]:
import pandas as pd
import numpy as np

df = dfCongreso.copy()

# -------------------------
# Funciones
# -------------------------
def fmt_mean_sd(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty: return "—"
    sd = s.std(ddof=1)
    if pd.isna(sd): sd = 0.0
    return f"{s.mean():.2f} ({sd:.2f})"

def fmt_mean(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return "—" if s.empty else f"{s.mean():.2f}"

def pct_times_total(d, pct_col, total_col="casos"):
    pct = pd.to_numeric(d.get(pct_col), errors="coerce")
    tot = pd.to_numeric(d.get(total_col), errors="coerce")
    if pct is None or tot is None:
        return pd.Series(np.nan, index=d.index)
    scale_0_100 = pct.dropna().median() > 1 if pct.notna().any() else False
    prop = pct / 100 if scale_0_100 else pct
    return prop * tot

# -------------------------
# Variables
# -------------------------
rows = [
    ("Subtópicos Upward (UW; prom.)", "Subtopicos_UW_prom", "mean"),
    ("Subtópicos Downward (DW; prom.)", "Subtopicos_DW_prom", "mean"),
    ("Tuw (casos arriba)", "casos_comparacion_arriba", "mean_sd"),
    ("Tdw (casos abajo)", "casos_comparacion_abajo", "mean_sd"),
    ("PropTup", "porce_topicos_arriba", "mean_sd"),
    ("PropTdown", "porce_topicos_abajo", "mean_sd"),
    ("PrefTcomp", "preferencia_casos_comparacion", "mean_sd"),
    ("PrefTup", "preferencia_casos_uw", "mean_sd"),
    ("Steps", "pasos_reales_ejercicio", "mean_sd"),
    ("Att1c", "steps_primera", "mean_sd"),
    ("PerfAtt", "performance", "mean_sd"),
    ("PerfAttup", "performance_arriba", "mean_sd"),
    ("PerfAttdw", "performance_abajo", "mean_sd"),
]

# -------------------------
# Construir columnas derivadas
# -------------------------
df["Subtopicos_UW_prom"] = pct_times_total(df, "porce_topicos_arriba", "casos")
df["Subtopicos_DW_prom"] = pct_times_total(df, "porce_topicos_abajo", "casos")

# -------------------------
# Tabla final
# -------------------------
orden_grupos = ["OLM", "OSLM", "OSLM+VP"]
tabla = pd.DataFrame(index=[r[0] for r in rows], columns=orden_grupos).fillna("—")

for g, sub in df.groupby("grupo_intervencion"):
    if g not in tabla.columns:
        continue
    for label, col, how in rows:
        if col not in sub.columns:
            continue
        tabla.loc[label, g] = (fmt_mean(sub[col]) if how == "mean" else fmt_mean_sd(sub[col]))

print("\nP3.1 y P3.2 (dfCongreso) — resumen por grupo (Steps = pasos_reales_ejercicio)")
print(tabla)



P3.1 y P3.2 (dfCongreso) — resumen por grupo (Steps = pasos_reales_ejercicio)
                                           OLM           OSLM        OSLM+VP
Subtópicos Upward (UW; prom.)             0.00           6.21           5.36
Subtópicos Downward (DW; prom.)           0.00           8.37           5.60
Tuw (casos arriba)                 0.00 (0.00)   6.21 (13.09)   5.36 (12.12)
Tdw (casos abajo)                  0.00 (0.00)   8.37 (11.59)    5.60 (9.88)
PropTup                            0.00 (0.00)    0.14 (0.23)    0.15 (0.25)
PropTdown                          0.00 (0.00)    0.37 (0.37)    0.24 (0.33)
PrefTcomp                         -1.00 (0.00)    0.03 (0.86)   -0.22 (0.86)
PrefTup                                      —   -0.45 (0.64)   -0.24 (0.66)
Steps                            58.19 (74.78)  60.60 (61.39)  56.21 (66.94)
Att1c                            56.19 (74.14)  58.56 (60.00)  53.77 (64.33)
PerfAtt                            0.94 (0.13)    0.97 (0.05)    0.96 (0.0

# PREGUNTA 2

# MIXEDLM - GENERAL

In [21]:
import pandas as pd
import statsmodels.formula.api as smf

formula = ("pollValue_next ~ pollValue + steps_primera + last_progressMe + "
           "selectionRating_result + comparacionOSLM + comparacionOSLMVP + "
           "proporcion_pasos_primera_vez")

num = ["pollValue_next","pollValue","steps_primera","last_progressMe",
       "selectionRating_result","proporcion_pasos_primera_vez"]

df = (dfSecuencia
      .dropna(subset=["pollValue_next","selectionRating_result"])
      .assign(
          comparacionOSLM=lambda d: (d["grupo_intervencion"] != "OLM").astype(int),
          comparacionOSLMVP=lambda d: (d["grupo_intervencion"] == "OSLM+VP").astype(int),
          **{c: (lambda d, c=c: pd.to_numeric(d[c], errors="coerce")) for c in num}
      )
      .loc[:, ["userid"] + num + ["comparacionOSLM","comparacionOSLMVP"]]
      .dropna()
      .assign(userid=lambda d: d["userid"].astype("category"))
      .reset_index(drop=True)
)

print(smf.mixedlm(formula, df, groups=df["userid"]).fit().summary())


                 Mixed Linear Model Regression Results
Model:                MixedLM     Dependent Variable:     pollValue_next
No. Observations:     449         Method:                 REML          
No. Groups:           80          Scale:                  211.9551      
Min. group size:      1           Log-Likelihood:         -1861.3746    
Max. group size:      28          Converged:              Yes           
Mean group size:      5.6                                               
------------------------------------------------------------------------
                             Coef.  Std.Err.   z    P>|z|  [0.025 0.975]
------------------------------------------------------------------------
Intercept                    23.883    6.694  3.568 0.000  10.764 37.003
pollValue                     0.503    0.031 16.460 0.000   0.443  0.563
steps_primera                 0.979    0.274  3.579 0.000   0.443  1.516
last_progressMe               7.292    3.515  2.075 0.038   0.403 14.

# MIXEDLM - D1-D2

In [22]:
import pandas as pd
import statsmodels.formula.api as smf

formula = ("pollValue_next ~ pollValue + steps_primera + last_progressMe + "
           "selectionRating_result + comparacionOSLM + comparacionOSLMVP + "
           "proporcion_pasos_primera_vez")

num = ["pollValue_next","pollValue","steps_primera","last_progressMe",
       "selectionRating_result","proporcion_pasos_primera_vez"]

df = (df_1_to_2
      .dropna(subset=["pollValue_next"])
      .assign(
          comparacionOSLM=lambda d: (d["grupo_intervencion"] != "OLM").astype(int),
          comparacionOSLMVP=lambda d: (d["grupo_intervencion"] == "OSLM+VP").astype(int),
          **{c: (lambda d, c=c: pd.to_numeric(d[c], errors="coerce")) for c in num}
      )
      .loc[:, ["userid"] + num + ["comparacionOSLM","comparacionOSLMVP"]]
      .dropna()
      .assign(userid=lambda d: d["userid"].astype("category"))
      .reset_index(drop=True)
)

print(smf.mixedlm(formula, df, groups=df["userid"]).fit().summary())


                  Mixed Linear Model Regression Results
Model:                 MixedLM     Dependent Variable:     pollValue_next
No. Observations:      233         Method:                 REML          
No. Groups:            77          Scale:                  201.0707      
Min. group size:       1           Log-Likelihood:         -966.1561     
Max. group size:       10          Converged:              Yes           
Mean group size:       3.0                                               
-------------------------------------------------------------------------
                              Coef.  Std.Err.   z    P>|z|  [0.025 0.975]
-------------------------------------------------------------------------
Intercept                     21.786    8.626  2.526 0.012   4.879 38.693
pollValue                      0.492    0.043 11.513 0.000   0.409  0.576
steps_primera                  1.341    0.364  3.683 0.000   0.627  2.054
last_progressMe               10.520    4.789  2.197 0.0

# MIXEDLM - D2-D3

In [23]:
import pandas as pd
import statsmodels.formula.api as smf

FORMULA = ("pollValue_next ~ pollValue + steps_primera + last_progressMe + "
           "selectionRating_result + comparacionOSLM + comparacionOSLMVP + "
           "proporcion_pasos_primera_vez")

NUM = ["pollValue_next","pollValue","steps_primera","last_progressMe",
       "selectionRating_result","proporcion_pasos_primera_vez"]

df = (df_2_to_3
      .dropna(subset=["pollValue_next"])
      .assign(
          comparacionOSLM=lambda d: (d["grupo_intervencion"] != "OLM").astype(int),
          comparacionOSLMVP=lambda d: (d["grupo_intervencion"] == "OSLM+VP").astype(int),
          **{c: (lambda d, c=c: pd.to_numeric(d[c], errors="coerce")) for c in NUM}
      )
      .loc[:, ["userid"] + NUM + ["comparacionOSLM","comparacionOSLMVP"]]
      .dropna()
      .assign(userid=lambda d: d["userid"].astype("category"))
      .reset_index(drop=True)
)

print(smf.mixedlm(FORMULA, df, groups=df["userid"]).fit().summary())


                 Mixed Linear Model Regression Results
Model:                MixedLM     Dependent Variable:     pollValue_next
No. Observations:     96          Method:                 REML          
No. Groups:           49          Scale:                  192.0841      
Min. group size:      1           Log-Likelihood:         -383.9425     
Max. group size:      8           Converged:              Yes           
Mean group size:      2.0                                               
------------------------------------------------------------------------
                             Coef.  Std.Err.   z    P>|z|  [0.025 0.975]
------------------------------------------------------------------------
Intercept                     2.081   13.783  0.151 0.880 -24.933 29.095
pollValue                     0.707    0.065 10.823 0.000   0.579  0.836
steps_primera                 0.486    0.586  0.829 0.407  -0.663  1.635
last_progressMe               7.020    7.506  0.935 0.350  -7.691 21.

# Análisis Diferencias

In [24]:
import pandas as pd

df = dfSecuencia[dfSecuencia['pollCount'].isin([1, 2, 3])].copy()

df_wide = (
    df.pivot_table(
        index=["userid", "grupo_intervencion"],
        columns="pollCount",
        values="pollValue"
    )
    .rename(columns={1: "poll1", 2: "poll2", 3: "poll3"})
    .reset_index()
)

df_wide["diff_1_2"] = df_wide["poll2"] - df_wide["poll1"]
df_wide["diff_2_3"] = df_wide["poll3"] - df_wide["poll2"]
df_wide["diff_1_3"] = df_wide["poll3"] - df_wide["poll1"]

resultados = (
    df_wide
    .groupby("grupo_intervencion")[["diff_1_2", "diff_2_3", "diff_1_3"]]
    .mean()
    .reset_index()
)

print(resultados)


pollCount grupo_intervencion   diff_1_2   diff_2_3   diff_1_3
0                        OLM  13.513919   3.218648  14.737500
1                       OSLM  12.045031   4.474717  20.350321
2                    OSLM+VP  13.909385  12.496941  27.946229


In [25]:
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

resultados_anova = {}

for diff in ["diff_1_2", "diff_2_3", "diff_1_3"]:
    print(f"\n===== {diff} =====")

    # ---------- ANOVA de un factor ----------
    model = ols(f"{diff} ~ C(grupo_intervencion)", data=df_wide).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print("\nANOVA:")
    print(anova_table)

    # ---------- Kruskal-Wallis ----------
    grupos = [
        df_wide.loc[df_wide["grupo_intervencion"] == g, diff].dropna()
        for g in df_wide["grupo_intervencion"].unique()
    ]
    H, p = stats.kruskal(*grupos)
    print("\nKruskal–Wallis:")
    print(f"H = {H:.3f}, p = {p:.4f}")


===== diff_1_2 =====

ANOVA:
                             sum_sq    df         F    PR(>F)
C(grupo_intervencion)     52.876039   2.0  0.122842  0.884572
Residual               17002.420728  79.0       NaN       NaN

Kruskal–Wallis:
H = 0.171, p = 0.9182

===== diff_2_3 =====

ANOVA:
                             sum_sq    df         F    PR(>F)
C(grupo_intervencion)    993.539013   2.0  1.643397  0.202508
Residual               16927.799427  56.0       NaN       NaN

Kruskal–Wallis:
H = 2.868, p = 0.2383

===== diff_1_3 =====

ANOVA:
                             sum_sq    df         F    PR(>F)
C(grupo_intervencion)   1736.425989   2.0  1.556157  0.219929
Residual               31243.578365  56.0       NaN       NaN

Kruskal–Wallis:
H = 3.661, p = 0.1603


# Pregunta 3.1

In [26]:
import pandas as pd
import numpy as np
from scipy import stats

G3 = ["OLM", "OSLM", "OSLM+VP"]
G2 = ["OSLM", "OSLM+VP"]

VARS_P31 = {  
    "PropTup":   "porce_topicos_arriba",
    "PropTdown": "porce_topicos_abajo",
    "PrefTcomp": "preferencia_casos_comparacion",
    "PrefTup":   "preferencia_casos_uw",
}

PROG_IND = "progreso_inicial"        # ANOVA 3 grupos
PROG_GRP = "progreso_inicial_grupo"  # MW 2 grupos

# =========================
# Datos (filtrados)
# =========================
df_all = dfCongreso.loc[dfCongreso["grupo_intervencion"].isin(G3)].copy()
df_2g  = df_all.loc[df_all["grupo_intervencion"].isin(G2)].copy()

# =========================
# Funciones
# =========================
to_num = lambda s: pd.to_numeric(s, errors="coerce").dropna()

def mean_sd_str(s):
    s = to_num(s)
    if s.empty: return "—"
    sd = s.std(ddof=1)
    if pd.isna(sd): sd = 0.0
    return f"{s.mean():.2f} ({sd:.2f})"

def mean_str(s):
    s = to_num(s)
    return "—" if s.empty else f"{s.mean():.2f}"

def mw(a, b):
    a, b = to_num(a), to_num(b)
    if a.empty or b.empty: return (np.nan, np.nan)
    try:
        U, p = stats.mannwhitneyu(a, b, alternative="two-sided", method="auto")
    except TypeError:
        U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    return (U, p)

def anova(*groups):
    gs = [to_num(g).values for g in groups]
    if any(len(g) == 0 for g in gs): return (np.nan, np.nan, [len(g) for g in gs])
    F, p = stats.f_oneway(*gs)
    return (F, p, [len(g) for g in gs])

def normality_rows(df, var, groups):
    out = []
    for g in groups:
        s = to_num(df.loc[df["grupo_intervencion"] == g, var])
        n = len(s)

        # Shapiro (>=3), cap 5000 como antes
        if n >= 3:
            s_sh = s.sample(min(n, 5000), random_state=123) if n > 5000 else s
            W, pW = stats.shapiro(s_sh)
        else:
            W, pW = np.nan, np.nan

        # D’Agostino (>=8)
        K2, pK2 = stats.normaltest(s) if n >= 8 else (np.nan, np.nan)

        # QQ corr (>=3)
        if n >= 3:
            osm, osr = stats.probplot(s, dist="norm")[0]
            rqq = np.corrcoef(osm, osr)[0, 1]
        else:
            rqq = np.nan

        out.append({"Variable": var, "Grupo": g, "n": n,
                    "Shapiro_W": W, "Shapiro_p": pW,
                    "Normaltest_K2": K2, "Normaltest_p": pK2,
                    "QQ_corr_r": rqq})
    return pd.DataFrame(out)

sig = lambda p: "*" if (pd.notna(p) and p < 0.05) else ""

# =========================
# 1) Table 4 (OSLM vs OSLM+VP) — SOLO P3.1
# =========================
tabla4 = pd.DataFrame({g: {lab: mean_sd_str(df_2g.loc[df_2g["grupo_intervencion"] == g, col])
                          for lab, col in VARS_P31.items()}
                       for g in G2})

# =========================
# 2) Tests principales
# =========================
mw_tests = {
    lab: mw(df_2g.loc[df_2g["grupo_intervencion"] == "OSLM", col],
            df_2g.loc[df_2g["grupo_intervencion"] == "OSLM+VP", col])
    for lab, col in VARS_P31.items()
}

F_progInd, p_progInd, ns_progInd = anova(
    df_all.loc[df_all["grupo_intervencion"] == "OLM", PROG_IND],
    df_all.loc[df_all["grupo_intervencion"] == "OSLM", PROG_IND],
    df_all.loc[df_all["grupo_intervencion"] == "OSLM+VP", PROG_IND],
)

U_progGrp, p_progGrp = mw(
    df_all.loc[df_all["grupo_intervencion"] == "OSLM", PROG_GRP],
    df_all.loc[df_all["grupo_intervencion"] == "OSLM+VP", PROG_GRP],
)

# =========================
# 3) Normalidad (supuestos)
# =========================
norm_anova = normality_rows(df_all, PROG_IND, G3)
norm_2g = pd.concat(
    [normality_rows(df_all, v, G2) for v in list(VARS_P31.values()) + [PROG_GRP]],
    ignore_index=True
)

# =========================
# 4) Salida
# =========================
n_oslm = df_2g.loc[df_2g["grupo_intervencion"] == "OSLM", "userid"].nunique()
n_vp   = df_2g.loc[df_2g["grupo_intervencion"] == "OSLM+VP", "userid"].nunique()

print("\nTable 4. Proporciones y preferencias en selección de subtópicos con comparación social")
print(f"OSLM (N={n_oslm}) vs OSLM+VP (N={n_vp})\n")
print(tabla4)

for lab, (U, p) in mw_tests.items():
    print(f"Mann–Whitney U {lab} (OSLM vs OSLM+VP): U = {U:.1f}, p = {p:.3f} {sig(p)}")

k = 3
N_anova = sum(ns_progInd) if all(isinstance(x, (int, np.integer)) for x in ns_progInd) else np.nan
df1, df2 = k - 1, (N_anova - k) if pd.notna(N_anova) else np.nan
print(f"\nANOVA ProgInd (OLM, OSLM, OSLM+VP): F({df1}, {df2}) = {F_progInd:.3f}, p = {p_progInd:.3f}")
print(f"Mann–Whitney U ProgGrp (OSLM vs OSLM+VP): U = {U_progGrp:.1f}, p = {p_progGrp:.3f}")

print("\n========================")
print("Chequeo de normalidad (supuestos)")
print("========================")

print("\n-- Para ANOVA (ProgInd) por grupo --")
print(norm_anova[["Variable","Grupo","n","Shapiro_W","Shapiro_p","Normaltest_K2","Normaltest_p","QQ_corr_r"]])

print("\n-- Para comparación OSLM vs OSLM+VP (variables clave) --")
print(norm_2g[["Variable","Grupo","n","Shapiro_W","Shapiro_p","Normaltest_K2","Normaltest_p","QQ_corr_r"]])



Table 4. Proporciones y preferencias en selección de subtópicos con comparación social
OSLM (N=52) vs OSLM+VP (N=70)

                   OSLM       OSLM+VP
PropTup     0.14 (0.23)   0.15 (0.25)
PropTdown   0.37 (0.37)   0.24 (0.33)
PrefTcomp   0.03 (0.86)  -0.22 (0.86)
PrefTup    -0.45 (0.64)  -0.24 (0.66)
Mann–Whitney U PropTup (OSLM vs OSLM+VP): U = 1795.5, p = 0.887 
Mann–Whitney U PropTdown (OSLM vs OSLM+VP): U = 2241.5, p = 0.023 *
Mann–Whitney U PrefTcomp (OSLM vs OSLM+VP): U = 2130.5, p = 0.095 
Mann–Whitney U PrefTup (OSLM vs OSLM+VP): U = 548.0, p = 0.132 

ANOVA ProgInd (OLM, OSLM, OSLM+VP): F(2, 124) = 0.097, p = 0.907
Mann–Whitney U ProgGrp (OSLM vs OSLM+VP): U = 916.0, p = 0.362

Chequeo de normalidad (supuestos)

-- Para ANOVA (ProgInd) por grupo --
           Variable    Grupo   n  Shapiro_W  Shapiro_p  Normaltest_K2  \
0  progreso_inicial      OLM  46   0.827182   0.000008       8.514076   
1  progreso_inicial     OSLM  39   0.859167   0.000181       5.373700   
2  pro

# PREGUNTA 3.2

In [27]:
import pandas as pd
import numpy as np
from scipy import stats

G3 = ["OLM", "OSLM", "OSLM+VP"]
G2 = ["OSLM", "OSLM+VP"]

COLS = {
    "PerfAtt":   "performance",
    "PerfAttup": "performance_arriba",
    "PerfAttdw": "performance_abajo",
}

df = dfCongreso.loc[dfCongreso["grupo_intervencion"].isin(G3)].copy()
missing = [c for c in COLS.values() if c not in df.columns]
if missing:
    raise KeyError(f"Faltan columnas en dfCongreso: {missing}")

to_num = lambda s: pd.to_numeric(s, errors="coerce").dropna()

def mean_sd_str(s):
    s = to_num(s)
    if s.empty: return "—"
    sd = s.std(ddof=1)
    if pd.isna(sd): sd = 0.0
    return f"{s.mean():.2f} ({sd:.2f})"

def mw(a, b):
    a, b = to_num(a), to_num(b)
    if a.empty or b.empty: return (np.nan, np.nan)
    try:
        return stats.mannwhitneyu(a, b, alternative="two-sided", method="auto")[:2]
    except TypeError:
        return stats.mannwhitneyu(a, b, alternative="two-sided")[:2]

def shapiro(s):
    s = to_num(s)
    n = len(s)
    if n < 3: return (n, np.nan, np.nan)
    if n > 5000: s = s.sample(5000, random_state=123)
    W, p = stats.shapiro(s)
    return (len(s), W, p)

# -------------------------
# Tabla (medias/SD)
# -------------------------
tabla = pd.DataFrame(index=COLS.keys(), columns=G3).fillna("—")

for g in G3:
    tabla.loc["PerfAtt", g] = mean_sd_str(df.loc[df["grupo_intervencion"] == g, COLS["PerfAtt"]])

for g in G2:
    for k in ["PerfAttup", "PerfAttdw"]:
        tabla.loc[k, g] = mean_sd_str(df.loc[df["grupo_intervencion"] == g, COLS[k]])

# -------------------------
# Contrastes
# -------------------------
arrs = [to_num(df.loc[df["grupo_intervencion"] == g, COLS["PerfAtt"]]).values for g in G3]
H, pH = (np.nan, np.nan) if any(len(a) == 0 for a in arrs) else stats.kruskal(*arrs)

U_up, p_up = mw(df.loc[df["grupo_intervencion"] == "OSLM",    COLS["PerfAttup"]],
                df.loc[df["grupo_intervencion"] == "OSLM+VP", COLS["PerfAttup"]])

U_dw, p_dw = mw(df.loc[df["grupo_intervencion"] == "OSLM",    COLS["PerfAttdw"]],
                df.loc[df["grupo_intervencion"] == "OSLM+VP", COLS["PerfAttdw"]])

# -------------------------
# Normalidad (Shapiro)
# -------------------------
norm = []
for g in G3:
    n, W, pW = shapiro(df.loc[df["grupo_intervencion"] == g, COLS["PerfAtt"]])
    norm.append({"Variable":"PerfAtt","Grupo":g,"n":n,"Shapiro_W":W,"Shapiro_p":pW})
for g in G2:
    for k in ["PerfAttup","PerfAttdw"]:
        n, W, pW = shapiro(df.loc[df["grupo_intervencion"] == g, COLS[k]])
        norm.append({"Variable":k,"Grupo":g,"n":n,"Shapiro_W":W,"Shapiro_p":pW})
norm_df = pd.DataFrame(norm)

# -------------------------
# Print
# -------------------------
Ns = {g: df.loc[df["grupo_intervencion"] == g, "userid"].nunique() for g in G3}
print("\nTabla. Performance en actividad (global y por dirección de comparación)")
print(f"OLM (N={Ns['OLM']}) | OSLM (N={Ns['OSLM']}) | OSLM+VP (N={Ns['OSLM+VP']})\n")
print(tabla)

print("\nContrastes")
print(f"Kruskal–Wallis PerfAtt (OLM, OSLM, OSLM+VP): H = {H:.2f}, p = {pH:.3f}")
print(f"Mann–Whitney U PerfAttup (OSLM vs OSLM+VP): U = {U_up:.1f}, p = {p_up:.3f}")
print(f"Mann–Whitney U PerfAttdw (OSLM vs OSLM+VP): U = {U_dw:.1f}, p = {p_dw:.3f}")

print("\nChequeo de normalidad (Shapiro–Wilk)")
print(norm_df)



Tabla. Performance en actividad (global y por dirección de comparación)
OLM (N=59) | OSLM (N=52) | OSLM+VP (N=70)

                   OLM         OSLM      OSLM+VP
PerfAtt    0.94 (0.13)  0.97 (0.05)  0.96 (0.07)
PerfAttup            —  0.33 (0.46)  0.33 (0.46)
PerfAttdw            —  0.62 (0.46)  0.43 (0.48)

Contrastes
Kruskal–Wallis PerfAtt (OLM, OSLM, OSLM+VP): H = 1.32, p = 0.517
Mann–Whitney U PerfAttup (OSLM vs OSLM+VP): U = 1839.0, p = 0.909
Mann–Whitney U PerfAttdw (OSLM vs OSLM+VP): U = 2201.0, p = 0.034

Chequeo de normalidad (Shapiro–Wilk)
    Variable    Grupo   n  Shapiro_W     Shapiro_p
0    PerfAtt      OLM  59   0.517160  1.215794e-12
1    PerfAtt     OSLM  52   0.676382  1.980475e-09
2    PerfAtt  OSLM+VP  65   0.639421  2.311023e-11
3  PerfAttup     OSLM  52   0.622695  2.584042e-10
4  PerfAttdw     OSLM  52   0.667233  1.378808e-09
5  PerfAttup  OSLM+VP  70   0.620499  3.636250e-12
6  PerfAttdw  OSLM+VP  70   0.676725  3.923895e-11
